# EAPS 507 — Applying Functions to Xarray Data
## Vectorization, broadcasting, and `xr.apply_ufunc()`

**Estimated time:** 90 minutes  
**Context:** atmospheric temperature on pressure levels

### Learning objectives

By the end of this tutorial, you should be able to:

1. explain vectorization and broadcasting in plain language;
2. distinguish elementwise operations from operations that use a core dimension;
3. write a numerical function that works on array-like inputs;
4. use `xr.apply_ufunc()` while preserving Xarray coordinates and metadata;
5. use `input_core_dims` and `output_core_dims` for profile-based calculations;
6. decide when direct Xarray arithmetic, `apply_ufunc()`, or `map_blocks()` is appropriate.

### Class rhythm

Each section follows **explain → predict → run → interpret → your turn**. Work with a partner for the prediction and “your turn” prompts, but write your own explanation.

| Section | Topic | Time |
|---|---|---:|
| 1 | Setup and labeled data | 10 min |
| 2 | Elementwise functions | 15 min |
| 3 | Broadcasting and vectorization | 15 min |
| 4 | `apply_ufunc()` | 20 min |
| 5 | Core dimensions | 20 min |
| 6 | 2-minute example + wrap-up | 10 min |


## 1. Setup and labeled data — 10 minutes

We will first try to open the ERA5 file used in class. If it is unavailable, the next cell constructs a small dataset with the same essential dimensions. This makes the tutorial portable while keeping the atmospheric-science context.


In [25]:
from pathlib import Path

import numpy as np
import xarray as xr

xr.set_options(keep_attrs=True, display_expand_data=False)

DATA_PATH = Path("/home/nkreigh/Desktop/EAPS_507/Students/ERA5_T_Rh_202504.nc")


In [26]:
if DATA_PATH.exists():
    temp_rh_main = (
        xr.open_mfdataset(DATA_PATH)#, chunks={'valid_time':1,
                                           # 'pressure_level':27})
        #renames variables for ease
        .rename(
            {
                "valid_time": "time", 
                "pressure_level": "level",
                "latitude": "lat",
                "longitude": "lon",
            }
        )
        .isel(time=slice(0, 5))#.chunks({'time':=1,})
        
        #data is stored in chunks, stored in memory, we want to load all the data so .load.
        
        .load()
    )
    print("Using the ERA5 class file.")
    #else creates temperature and humidity data for us.
else:
    rng = np.random.default_rng(507)
    time = np.arange("2025-04-01T00", "2025-04-01T12", dtype="datetime64[3h]")
    level = np.array([1000, 925, 850, 700, 500, 300])
    lat = np.array([39.5, 40.0, 40.5])
    lon = np.array([-87.5, -87.0, -86.5, -86.0])

    base_profile = 292 - 0.065 * (1000 - level)
    temperature = (
        base_profile[None, :, None, None]
        + rng.normal(0, 1.2, size=(time.size, level.size, lat.size, lon.size))
    )
    relative_humidity = rng.uniform(
        35, 95, size=(time.size, level.size, lat.size, lon.size)
    )

    temp_rh_main = xr.Dataset(
        data_vars={
            "t": (
                ("time", "level", "lat", "lon"),
                temperature,
                {"long_name": "air temperature", "units": "K"},
            ),
            "r": (
                ("time", "level", "lat", "lon"),
                relative_humidity,
                {"long_name": "relative humidity", "units": "%"},
            ),
        },
        coords={"time": time, "level": level, "lat": lat, "lon": lon},
        attrs={"description": "Synthetic EAPS 507 atmospheric dataset"},
    )
    print("Class file not found; using the portable sample dataset.")

temp_rh_main


Using the ERA5 class file.


<xarray.Dataset> Size: 78MB
Dimensions:  (time: 5, level: 27, lat: 201, lon: 361)
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Data variables:
    r        (time, level, lat, lon) float32 39MB 34.35 31.89 ... 52.28 52.84
    t        (time, level, lat, lon) float32 39MB 277.1 277.5 ... 195.0 194.9
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-29T00:00 GRIB to CDM+CF via cfgrib-0.9.1...

### Read the data model

An Xarray `DataArray` combines values with:

- **dimensions**: axis names such as `time`, `level`, `lat`, and `lon`;
- **coordinates**: labels along those dimensions;
- **attributes**: descriptive metadata such as units.

The labels are central to how Xarray aligns and broadcasts calculations.


In [27]:
list(temp_rh_main.variables)

['r', 't', 'number', 'time', 'level', 'lat', 'lon', 'expver']

In [28]:
t = temp_rh_main["t"]

print("Dimensions:", t.dims)
print("Shape:", t.shape)
print("Coordinates:", list(t.coords))
print("Units:", t.attrs.get("units"))


Dimensions: ('time', 'level', 'lat', 'lon')
Shape: (5, 27, 201, 361)
Coordinates: ['number', 'time', 'level', 'lat', 'lon', 'expver']
Units: K


### Predict, then test

Before running the next cell, predict the result of selecting one time and one location.

1. Which dimension will remain?
2. What will the shape be?
3. Will the pressure-level labels be preserved?


In [29]:
profile = t.isel(time=0, lat=1, lon=1)
profile


<xarray.DataArray 't' (level: 27)> Size: 108B
277.3 275.9 274.5 273.2 271.8 270.3 ... 225.4 225.9 225.5 225.5 224.5 225.2
Coordinates:
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
    number   int64 8B 0
    time     datetime64[ns] 8B 2025-04-01
    lat      float64 8B 59.75
    lon      float64 8B -129.8
    expver   <U4 16B '0001'
Attributes: (12/31)
    GRIB_paramId:                             130
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      72561
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                Temperature
    units:                                    K
    standard_name:                            air_temperature

### Your turn 1

Select the temperature profile at the **last time**, nearest to **40°N, 87°W**. Use `.sel(..., method="nearest")` for latitude and longitude.


In [30]:
last_time = temp_rh_main.time.values[-1]

In [31]:
# YOUR TURN 1
# selected_profile = ...
# selected_profile

#selects temperature profile for the selected latitude and longitude
selected_profile = temp_rh_main.sel(time = last_time,
                                    lon =-87, lat = 40,method = 'nearest')

selected_profile

<xarray.Dataset> Size: 480B
Dimensions:  (level: 27)
Coordinates:
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
    number   int64 8B 0
    time     datetime64[ns] 8B 2025-04-01T04:00:00
    lat      float64 8B 40.0
    lon      float64 8B -87.0
    expver   <U4 16B '0001'
Data variables:
    r        (level) float32 108B 73.23 68.62 68.49 74.36 ... 2.098 2.003 2.856
    t        (level) float32 108B 276.6 275.2 274.3 273.4 ... 221.6 220.2 215.8
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-29T00:00 GRIB to CDM+CF via cfgrib-0.9.1...

---
## 2. Elementwise functions — 15 minutes

An **elementwise function** treats every value independently. Converting Kelvin to Celsius is elementwise: each output value depends on one input value at the same position.

NumPy-style arithmetic is already vectorized. We describe the calculation once, and the array library applies it across all values without an explicit Python loop.


In [32]:
def kelvin_to_celsius(temperature_k):
    'Convert an array-like temperature from kelvin to degrees Celsius.'
    return temperature_k - 273.15

# pulls kelvin into celsius function and shows attributes
t_c = kelvin_to_celsius(t)
t_c.attrs = {"long_name": "air temperature", "units": "degC"}
t_c


<xarray.DataArray 't' (time: 5, level: 27, lat: 201, lon: 361)> Size: 39MB
3.989 4.364 4.657 4.79 4.729 4.551 ... -78.21 -78.18 -78.16 -78.16 -78.18 -78.21
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes:
    long_name:  air temperature
    units:      degC

### Why did that work without `apply_ufunc()`?

The function uses subtraction, which both NumPy and Xarray define for arrays. When a function is composed entirely of operations Xarray understands, calling it directly is usually the clearest choice.

> **Rule of thumb:** try direct Xarray arithmetic first. Reach for `apply_ufunc()` when you need to wrap a function that does not already understand labeled Xarray objects, or when you must explicitly describe core dimensions.


### Loop versus vectorized thinking

The two calculations below are conceptually equivalent, but the vectorized form is shorter, clearer, and generally faster.


In [33]:
small_values = np.array([280.0, 285.0, 290.0])

# Explicit Python loop
loop_result = []
for value in small_values:
    loop_result.append(value - 273.15)
loop_result = np.array(loop_result)

# Vectorized array expression
vectorized_result = small_values - 273.15

print("Loop result:      ", loop_result)
print("Vectorized result:", vectorized_result)
print("Same result?", np.allclose(loop_result, vectorized_result))


Loop result:       [ 6.85 11.85 16.85]
Vectorized result: [ 6.85 11.85 16.85]
Same result? True


### Student-led test 2

Write a function named `celsius_to_fahrenheit()` using:


$T_F = \frac{9}{5}T_C + 32.$

Apply it directly to `t_c`. Confirm that the output retains all four dimensions and the original coordinates.


In [34]:
# YOUR TURN 2
def celsius_to_fahrenheit(temperature_c): #convert an array like temperature from Celsius to Fahrenhiet'
    return temperature_c * (9 / 5 ) + 32
#we defined t_c earlier, so pull from that list. temperature_c is
t_f = celsius_to_fahrenheit(t_c)
t_c.attrs = {"long_name": "air_temperature", "units": "degF"}
#print(t_f.dims)
t_f

###def kelvin_to_celsius(temperature_k):
   # 'Convert an array-like temperature from kelvin to degrees Celsius.'
   # return temperature_k - 273.15


#t_c = kelvin_to_celsius(t)
#t_c.attrs = {"long_name": "air temperature", "units": "degC"}
#t_c###

<xarray.DataArray 't' (time: 5, level: 27, lat: 201, lon: 361)> Size: 39MB
39.18 39.86 40.38 40.62 40.51 40.19 ... -108.7 -108.7 -108.7 -108.7 -108.8
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes:
    long_name:  air temperature
    units:      degC

---
## 3. Broadcasting and vectorization — 15 minutes

**Broadcasting** lets arrays with different dimensions participate in one calculation when their dimensions are compatible.

Potential temperature is


$\theta = T\left(\frac{p_0}{p}\right)^{R_d/c_p}$,


where (T) varies over `time`, `level`, `lat`, and `lon`, while pressure (p) varies only over `level`.


In [35]:
P0_HPA = 1000.0
RD_OVER_CP = 0.286


def potential_temperature(temperature_k, pressure_hpa):
    'Potential temperature in K from temperature and pressure in hPa.'
    return temperature_k * (P0_HPA / pressure_hpa) ** RD_OVER_CP


pressure = temp_rh_main["level"]

print("Temperature dimensions:", t.dims)
print("Pressure dimensions:   ", pressure.dims)


Temperature dimensions: ('time', 'level', 'lat', 'lon')
Pressure dimensions:    ('level',)


### Predict first
###time, level, lat, long
What dimensions should the result have when we calculate `potential_temperature(t, pressure)`?

Xarray matches `pressure.level` to `t.level`, then broadcasts pressure across `time`, `lat`, and `lon`. It does not rely only on axis position.


In [36]:
theta_direct = potential_temperature(t, pressure)
theta_direct.name = "theta"
theta_direct.attrs = {"long_name": "potential temperature", "units": "K"}

print("Result dimensions:", theta_direct.dims)
print("Result shape:", theta_direct.shape)
theta_direct.isel(time=0, lat=1, lon=1)


Result dimensions: ('time', 'level', 'lat', 'lon')
Result shape: (5, 27, 201, 361)


<xarray.DataArray 'theta' (level: 27)> Size: 216B
277.3 277.9 278.6 279.3 280.1 280.8 ... 345.4 357.9 371.3 388.0 406.9 435.1
Coordinates:
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
    number   int64 8B 0
    time     datetime64[ns] 8B 2025-04-01
    lat      float64 8B 59.75
    lon      float64 8B -129.8
    expver   <U4 16B '0001'
Attributes:
    long_name:  potential temperature
    units:      K

### Coordinate alignment test

Xarray aligns inputs by coordinate labels. Reversing the order of the pressure coordinate should not change the physical answer, because the labels still identify each level.


In [37]:
pressure_reversed = pressure.isel(level=slice(None, None, -1))
theta_reversed_input = potential_temperature(t, pressure_reversed)

xr.testing.assert_allclose(theta_direct, theta_reversed_input)
print("Test passed: Xarray aligned pressure by level labels.")


Test passed: Xarray aligned pressure by level labels.


### Student-led test 3

Create a `DataArray` named `offset` with one value for each latitude: `[-1.0, 0.0, 1.0] degrees. Add it to `t`.

Before running your code, answer:

1. Across which dimensions will `offset` be broadcast?
2. What dimensions will the result have?
3. Why is naming the dimension `lat` important?


In [38]:
# YOUR TURN 3
# 1. it will be broadcast across all dimensions of t, so time, level, longitude, and latitude
# 2. the result will still have all the previous dimenstions of time, level, longitude, and latitude
# 3. naming the dimension lat will let xarray match coordinates after the offeset is applied
#defines the offset with -1 and 1 degree latitude

offset = xr.DataArray(
    [-1.0, 0.0, 1.0], 
    dims=['lat'], 
    coords={'lat': [-1.0, 0.0, 1.0]}
)

#add t to the offset
adjusted_t = t + offset
adjusted_t



<xarray.DataArray (time: 5, level: 27, lat: 0, lon: 361)> Size: 0B

Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 0B 
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes: (12/31)
    GRIB_paramId:                             130
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      72561
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                Temperature
    units:                                    K
    standard_name:                            air_temperature

---
## 4. Applying elementwise functions with `xr.apply_ufunc()` — 20 minutes

`xr.apply_ufunc()` adapts an array-oriented function to Xarray objects. It handles labeled alignment, broadcasting, and reconstruction of the output `DataArray`.

For this elementwise potential-temperature function, no core dimensions need to be declared.


In [39]:
theta = xr.apply_ufunc(
    potential_temperature,
    t,
    pressure,
    dask="parallelized", #dask is a manager that tries to manage cpu and nodes for whichever operation, parallelize makes it run in a shorter time
    output_dtypes=[float],
    keep_attrs=True,
)

theta.name = "theta"
theta.attrs.update(long_name="potential temperature", units="K")
theta


<xarray.DataArray 'theta' (time: 5, level: 27, lat: 201, lon: 361)> Size: 78MB
277.1 277.5 277.8 277.9 277.9 277.7 ... 376.6 376.7 376.7 376.7 376.7 376.6
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes: (12/31)
    GRIB_paramId:                             130
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      72561
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                potential temperature
    units:                                    K
    standard_name:                            air_temperature

### What each argument does

- `potential_temperature`: the function to call;
- `t, pressure`: the positional inputs passed to that function;
- `dask="parallelized"`: permits blockwise execution when inputs are Dask-backed;
- `output_dtypes=[float]`: tells Dask/Xarray the output data type;
- `keep_attrs=True`: asks Xarray to retain input attributes where possible.

`apply_ufunc()` preserves dimensions and coordinates, but scientific output metadata should still be set deliberately.


In [40]:
xr.testing.assert_allclose(theta, theta_direct)
print("Test passed: direct arithmetic and apply_ufunc give the same values.")


Test passed: direct arithmetic and apply_ufunc give the same values.


### A scalar-only function

Some legacy functions accept one scalar at a time. Setting `vectorize=True` asks `apply_ufunc()` to loop over broadcast dimensions. This is convenient, but it is usually slower than rewriting the function to accept arrays.


In [41]:
%time
import math


def scalar_heat_index_like(temperature_c, relative_humidity_pct):
    'Small teaching example that deliberately uses scalar math.'
    return temperature_c + 0.05 * relative_humidity_pct + math.sin(temperature_c / 10)


heat_metric = xr.apply_ufunc(
    scalar_heat_index_like,
    t_c,
    temp_rh_main["r"],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

heat_metric.name = "heat_metric"
heat_metric.attrs = {"long_name": "teaching-only heat metric", "units": "degC"}
heat_metric


CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.77 μs


<xarray.DataArray 'heat_metric' (time: 5, level: 27, lat: 201, lon: 361)> Size: 78MB
6.095 6.381 6.612 6.685 6.578 6.398 ... -76.59 -76.57 -76.56 -76.56 -76.57
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * level    (level) float64 216B 1e+03 975.0 950.0 925.0 ... 150.0 125.0 100.0
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes:
    long_name:  teaching-only heat metric
    units:      degC

### Student-led test 4

Write an array-aware function `temperature_anomaly(values, reference)` that subtracts a reference temperature. Apply it with `xr.apply_ufunc()` using:

- `values = t`;
- `reference = t.mean("time")`.

Then verify that the mean anomaly over `time` is approximately zero at every grid point.

**Question:** Why is `vectorize=True` unnecessary here?


In [42]:
# YOUR TURN 4

#vectorize=True is not necessary as xarray handles the arrays already

#define function for temperature anomoly
def temperature_anomaly(values, reference):
    return values - reference

# set anomaly by using xr.apply_ufunc, set it to output floats
anomaly = xr.apply_ufunc(
    temperature_anomaly, 
    t, 
    t.mean("time"), 
    dask="parallelized", 
    output_dtypes=[float]
)

#prints the absolute value and max mean anomoly. which is near 0 as expected.
print(float(np.abs(anomaly.mean("time")).max()))


3.967285010730848e-05


### Decision checkpoint

| Situation | Preferred approach |
|---|---|
| Formula already works with Xarray operations | Call it directly |
| NumPy function operates elementwise | `xr.apply_ufunc()` |
| Function consumes one or more named axes | `apply_ufunc()` with core dimensions |
| Scalar-only legacy function | `apply_ufunc(..., vectorize=True)` if rewriting is impractical |
| Function consumes/returns whole Xarray objects per chunk | `xr.map_blocks()` |


---
## 5. Functions that use a core dimension — 20 minutes

Some functions do not operate independently on every element. A vertical-profile calculation may consume the entire `level` dimension and return one value per profile.

In `apply_ufunc()` terminology, `level` is a **core dimension**. All other dimensions are broadcast dimensions.


In [43]:
def vertical_range(profile):
    'Return max minus min along the final axis of a profile.'
    return np.nanmax(profile, axis=-1) - np.nanmin(profile, axis=-1)


temperature_range = xr.apply_ufunc(
    vertical_range,
    t,
    input_core_dims=[["level"]],
    output_core_dims=[[]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

temperature_range.name = "vertical_temperature_range"
temperature_range.attrs = {
    "long_name": "vertical temperature range",
    "units": "K",
}

temperature_range


<xarray.DataArray 'vertical_temperature_range' (time: 5, lat: 201, lon: 361)> Size: 3MB
53.47 53.45 53.37 53.17 53.04 52.78 ... 101.8 101.8 101.8 101.9 101.9 102.0
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes:
    long_name:  vertical temperature range
    units:      K

### Read the core-dimension specification

```python
input_core_dims=[["level"]]
```

The function receives `level` as the final axis of each input profile.

```python
output_core_dims=[[]]
```

The output has no core dimension because one profile becomes one scalar. The broadcast dimensions `time`, `lat`, and `lon` remain.

`vectorize=True` applies the profile function separately to every combination of those broadcast coordinates.


### Predict first

If `t` has dimensions `(time, level, lat, lon)`, why does the output have `(time, lat, lon)` rather than `(time, level, lat, lon)`?

Discuss for one minute, then inspect the result below.


In [44]:
print("Input dimensions: ", t.dims)
print("Output dimensions:", temperature_range.dims)


Input dimensions:  ('time', 'level', 'lat', 'lon')
Output dimensions: ('time', 'lat', 'lon')


### A function with two profile inputs

The next function calculates a pressure-weighted mean temperature. Both temperature and pressure use `level` as a core dimension.


In [45]:
def pressure_weighted_mean(temperature_profile, pressure_profile):
    'Weighted mean over the final axis, ignoring missing temperature values.'
    valid = np.isfinite(temperature_profile)
    weights = np.where(valid, pressure_profile, 0.0)
    numerator = np.nansum(temperature_profile * weights, axis=-1)
    denominator = np.sum(weights, axis=-1)
    return numerator / denominator


weighted_mean_t = xr.apply_ufunc(
    pressure_weighted_mean,
    t,
    pressure,
    input_core_dims=[["level"], ["level"]],
    output_core_dims=[[]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
)

weighted_mean_t.name = "pressure_weighted_mean_temperature"
weighted_mean_t.attrs = {"long_name": "pressure-weighted mean temperature", "units": "K"}
weighted_mean_t


<xarray.DataArray 'pressure_weighted_mean_temperature' (time: 5, lat: 201,
                                                        lon: 361)> Size: 3MB
259.0 259.0 259.0 259.0 258.9 258.7 ... 278.1 278.1 278.1 278.0 278.0 278.0
Coordinates:
  * time     (time) datetime64[ns] 40B 2025-04-01 ... 2025-04-01T04:00:00
    expver   (time) <U4 80B '0001' '0001' '0001' '0001' '0001'
  * lat      (lat) float64 2kB 60.0 59.75 59.5 59.25 ... 10.75 10.5 10.25 10.0
  * lon      (lon) float64 3kB -130.0 -129.8 -129.5 ... -40.5 -40.25 -40.0
    number   int64 8B 0
Attributes:
    long_name:  pressure-weighted mean temperature
    units:      K

### Student-led test 5

Write a function `profile_mean(profile)` that returns the mean along its final axis. Use `xr.apply_ufunc()` to apply it across `level`.

Your output should retain `time`, `lat`, and `lon`, but not `level`. Compare your result with `t.mean("level")` using `xr.testing.assert_allclose()`.


In [46]:
# YOUR TURN 5

# defines function for the profile mean using numpy
def profile_mean(profile):
    return np.mean(profile, axis=-1)

# computes the mean temperature along a level
mean_with_ufunc = xr.apply_ufunc(
    profile_mean,
    t,
    input_core_dims=[["level"]],
    output_core_dims=[[]],
    dask="parallelized",
    output_dtypes=[float]
)

# renames the created mean
mean_with_ufunc.name = "temperature_profile_mean"
mean_with_ufunc.attrs = {"long_name": "mean temperature along level", "units": "K"}

#compares the results with t.mean('level')
xr.testing.assert_allclose(mean_with_ufunc, t.mean("level"))


### Common errors and how to diagnose them

1. **Unexpected dimensions:** inspect `.dims` for every input and output.
2. **Alignment creates missing values:** inspect coordinate labels; use `xr.align()` deliberately when needed.
3. **Core-dimension error with Dask:** a core dimension may need to be stored in one chunk, for example `.chunk({"level": -1})`.
4. **Wrong output dtype:** specify `output_dtypes` when using `dask="parallelized"`.
5. **Lost or misleading metadata:** assign the output name, units, and `long_name` explicitly.
6. **Slow `vectorize=True`:** rewrite scalar code using NumPy array operations where possible.


---
## 6. Two-minute example — 10 minutes

### Independent task

Create a function called `wind_speed(u, v)` that calculates

\[
V = \sqrt{u^2 + v^2}.
\]

The cell below supplies small `u` and `v` wind-component arrays. Apply your function with `xr.apply_ufunc()` and answer the interpretation questions.

**This is the part to submit:** export only Section 6 as a PDF, including your completed code, output, and written responses.


In [47]:
u = xr.DataArray(
    [[3.0, 0.0, -5.0], [6.0, 8.0, 0.0]],
    dims=("station", "time"),
    coords={"station": ["A", "B"], "time": ["00Z", "06Z", "12Z"]},
    attrs={"units": "m s-1", "long_name": "eastward wind"},
)

v = xr.DataArray(
    [[4.0, 5.0, 12.0], [8.0, 6.0, -7.0]],
    dims=("station", "time"),
    coords={"station": ["A", "B"], "time": ["00Z", "06Z", "12Z"]},
    attrs={"units": "m s-1", "long_name": "northward wind"},
)

display(u)
display(v)


<xarray.DataArray (station: 2, time: 3)> Size: 48B
3.0 0.0 -5.0 6.0 8.0 0.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
  * time     (time) <U3 36B '00Z' '06Z' '12Z'
Attributes:
    units:      m s-1
    long_name:  eastward wind

<xarray.DataArray (station: 2, time: 3)> Size: 48B
4.0 5.0 12.0 8.0 6.0 -7.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
  * time     (time) <U3 36B '00Z' '06Z' '12Z'
Attributes:
    units:      m s-1
    long_name:  northward wind

In [48]:
# SECTION 6 — COMPLETE AND SUBMIT THIS CELL

# 1. Define the function.
def wind_speed(u_component, v_component):
    return np.sqrt(u_component**2 + v_component**2)

# 2. Apply the function with xr.apply_ufunc().
speed = xr.apply_ufunc(
    wind_speed,
    u,
    v,
    dask="parallelized",
    output_dtypes=[float]
)

# 3. Add a name and scientifically appropriate metadata.
speed.name = "wind_speed"
speed.attrs = {"long_name": "wind speed", "units": "m s-1"}

# 4. Display the result.
speed


<xarray.DataArray 'wind_speed' (station: 2, time: 3)> Size: 48B
5.0 5.0 13.0 10.0 10.0 7.0
Coordinates:
  * station  (station) <U1 8B 'A' 'B'
  * time     (time) <U3 36B '00Z' '06Z' '12Z'
Attributes:
    long_name:  wind speed
    units:      m s-1

### Section 6 interpretation — include in your submission

Answer in complete sentences.

1. What are the dimensions and coordinates of `speed`?    
2. At which station and time is wind speed greatest? Give its value and units.
3. Why does this elementwise function not require `input_core_dims`?
4. Would `vectorize=True` be necessary for the function you wrote? Explain.


1. The dimensions of speed are station and time.
2. The greatest is at station A with 13m/s at 12Z.
3. Xarray already broadcasts the operation across the other dimensions, we do not have to tell it to do it manually.
4. No, as NumPy native array already supports elementwise functions.

## Wrap-up

- **Vectorization** expresses a calculation for arrays instead of writing a Python loop over individual values.
- **Broadcasting** expands compatible arrays across dimensions and, in Xarray, uses dimension names.
- **Direct Xarray arithmetic** is best when the function already supports labeled arrays.
- **`xr.apply_ufunc()`** adapts unlabeled array functions while restoring Xarray dimensions and coordinates.
- **Core dimensions** are the axes a function consumes as a unit, such as an entire vertical profile.

### Exit ticket

In one sentence each:

1. What is the difference between a broadcast dimension and a core dimension?
2. When would you use `vectorize=True`?
3. Why should metadata be checked after applying a custom function?


1. A core dimension takes all of one dimension and condenses it to something such as a mean, a broadcast creates a change for all values in a dimension.
2. You would use vectorize = True for any other core dimensions that NumPy or Xarray does not do for you automatically.
3. You need to check metadata after applying a function to make sure units or names were not altered, and if they are, you need to reassign them.